# Resume Screening System - Prototyping & Analysis

This notebook demonstrates the core logic of the Resume Screening System, including:
1. Data Loading & Exploration
2. Text Cleaning & Preprocessing
3. **New Feature**: Email & Phone Number Extraction
4. Skill Extraction
5. Similarity Calculation (Ranking)
6. **New Feature**: Interactive Visualizations (Plotly)

In [ ]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
from utils import clean_text, extract_skills, calculate_similarity, load_data

# Load Data
DATA_PATH = "data/resumes.csv"
df = load_data(DATA_PATH)
print(f"Loaded {len(df)} candidates.")
df.head()

## 1. Prototyping Contact Info Extraction
Developing Regex patterns to extract Email and Phone numbers from text.

In [ ]:
def extract_contact_info(text):
    text = str(text)
    # Email Regex
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    emails = re.findall(email_pattern, text)
    
    # Phone Regex (Supports various formats like +1-234-567-8900, (123) 456-7890, 123 456 7890)
    phone_pattern = r'\b(?:\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b'
    phones = re.findall(phone_pattern, text)
    
    return {
        "emails": ", ".join(set(emails)) if emails else "N/A",
        "phones": ", ".join(set(phones)) if phones else "N/A"
    }

# Test on a dummy string
dummy_text = "Contact me at john.doe@example.com or +1-555-0199. Also try jane_doe123@gmail.co.uk."
print("Extraction Test:", extract_contact_info(dummy_text))

In [ ]:
# Apply to DataFrame (Simulating extraction from 'job_description' or raw text if available)
# Note: In the real CSV, we might need a raw text column. For now, we'll try 'job_description'.
df['Contact Info'] = df['job_description'].apply(extract_contact_info)
df['Email'] = df['Contact Info'].apply(lambda x: x['emails'])
df['Phone'] = df['Contact Info'].apply(lambda x: x['phones'])

df[['job_title', 'Email', 'Phone']].head()

## 2. Screening Simulation
Simulating the screening process with a sample Job Description.

In [ ]:
job_description = """
We are looking for a Python Developer with experience in Machine Learning and SQL.
Must have knowledge of Pandas, Scikit-learn, and Cloud platforms like AWS.
"""

cleaned_jd = clean_text(job_description)
candidate_texts = df['job_description'].astype(str).tolist()
cleaned_candidates = [clean_text(text) for text in candidate_texts]

# Calculate Scores
scores = calculate_similarity(cleaned_jd, cleaned_candidates)
df['Match Score'] = [round(s * 100, 2) for s in scores]

# Rank
ranked_df = df.sort_values(by="Match Score", ascending=False).head(10).copy()
ranked_df[['job_title', 'Match Score', 'Email']].head()

## 3. Interactive Visualization (Plotly)
Prototyping charts for the dashboard.

In [ ]:
fig = px.bar(ranked_df, 
             x='job_title', 
             y='Match Score', 
             color='Match Score', 
             title='Top 10 Candidates Match Score',
             hover_data=['Email'],
             text='Match Score')
fig.show()

In [ ]:
# Skill Distribution (Prototype)
all_skills = []
for text in ranked_df['job_description']:
    all_skills.extend(extract_skills(str(text)))

skill_counts = pd.Series(all_skills).value_counts().reset_index()
skill_counts.columns = ['Skill', 'Count']

fig2 = px.pie(skill_counts, values='Count', names='Skill', title='Identified Skills across Top Candidates')
fig2.show()